# Python Machine Learning Labs: Book Rating Predictions

### Introduction

In this project, ***.

### Data Import

In [1]:
"""
Books ML Project
Analysis and machine learning on a dataset of books.
"""
import pandas as pd

In [2]:
books = pd.read_csv('books.csv', on_bad_lines='warn')
print(books.columns.tolist())
print(books.shape)

['bookID', 'title', 'authors', 'average_rating', 'isbn', 'isbn13', 'language_code', '  num_pages', 'ratings_count', 'text_reviews_count', 'publication_date', 'publisher']
(11123, 12)


C:\Users\guitt\AppData\Local\Temp\ipykernel_32148\2670025949.py:1: ParserWarning: Skipping line 3350: expected 12 fields, saw 13
Skipping line 4704: expected 12 fields, saw 13
Skipping line 5879: expected 12 fields, saw 13
Skipping line 8981: expected 12 fields, saw 13

  books = pd.read_csv('books.csv', on_bad_lines='warn')


In [3]:
books.columns = books.columns.str.strip() # remove extra spaces in column names, notably num_pages

Upon importing the data with `pd.read_csv('books.csv', on_bad_lines='warn')`, the data frame is created but we know 4 rows are skipped for having an extra column. Python says these are rows 3350, 4704, 5879, and 8981. Let's see what those lines look like. 

In [4]:
with open('books.csv', 'r', encoding='utf-8') as f:
    comma_lines = f.readlines()

# check the problematic lines (subtract 1 for 0-indexing)
for i in [3349, 4703, 5878, 8980]:
    print(f"Line {i+1}: {comma_lines[i]}")

Line 3350: 12224,Streetcar Suburbs: The Process of Growth in Boston  1870-1900,Sam Bass Warner, Jr./Sam B. Warner,3.58,0674842111,9780674842113,en-US,236,61,6,4/20/2004,Harvard University Press

Line 4704: 16914,The Tolkien Fan's Medieval Reader,David E. Smith (Turgon of TheOneRing.net, one of the founding members of this Tolkien website)/Verlyn Flieger/Turgon (=David E. Smith),3.58,1593600119,9781593600112,eng,400,26,4,4/6/2004,Cold Spring Press

Line 5879: 22128,Patriots (The Coming Collapse),James Wesley, Rawles,3.63,156384155X,9781563841552,eng,342,38,4,1/15/1999,Huntington House Publishers

Line 8981: 34889,Brown's Star Atlas: Showing All The Bright Stars With Full Instructions How To Find And Use Them For Navigational Purposes And Department Of Trade Examinations.,Brown, Son & Ferguson,0.00,0851742718,9780851742717,eng,49,0,0,5/1/1977,Brown Son & Ferguson Ltd.



We can see from this code the issue is due to an extra comma in the author column. It's much easier to see this issue when viewing the CSV in Excel, filtering this erroneous 13th column by all non-blank values to see these 4 rows. We can fix these rows manually and add them to our dataframe if we don't want to lose the data. 

In [5]:
bad_lines = [3349, 4703, 5878, 8980]
manual_rows = []
with open('books.csv', 'r', encoding='utf-8') as f:
    lines = f.readlines()
    for i in bad_lines:
        line = lines[i].strip()
        parts = line.split(',', 12)
        if len(parts) == 13:
            parts = [parts[0], parts[1], parts[2] + parts[3], parts[4], parts[5], parts[6], parts[7], parts[8], parts[9], parts[10], parts[11], parts[12]]
        manual_rows.append(parts)

manual_df = pd.DataFrame(manual_rows, columns=books.columns)
manual_df # ensure it has the correct data

,bookID,title,authors,average_rating,isbn,isbn13,language_code,num_pages,ratings_count,text_reviews_count,publication_date,publisher
0,12224,Streetcar Suburbs: The Process of Growth in Bo...,Sam Bass Warner Jr./Sam B. Warner,3.58,0674842111,9780674842113,en-US,236,61,6,4/20/2004,Harvard University Press
1,16914,The Tolkien Fan's Medieval Reader,David E. Smith (Turgon of TheOneRing.net one o...,3.58,1593600119,9781593600112,eng,400,26,4,4/6/2004,Cold Spring Press
2,22128,Patriots (The Coming Collapse),James Wesley Rawles,3.63,156384155X,9781563841552,eng,342,38,4,1/15/1999,Huntington House Publishers
3,34889,Brown's Star Atlas: Showing All The Bright Sta...,Brown Son & Ferguson,0.00,0851742718,9780851742717,eng,49,0,0,5/1/1977,Brown Son & Ferguson Ltd.


In [9]:
# We confirmed the data is correct, so we can concatonate manual_df to books.
books = pd.concat([books, manual_df], ignore_index=True)
print(books.shape) # Print shape to confirm it is the expected 11127 rows, 12 columns.

(11131, 12)


### Data Exploration & Cleaning

We begin exploring the data to get a better idea of how it's structured. 

In [ ]:
books.head() # General view of the data.

<class 'pandas.DataFrame'>
RangeIndex: 11131 entries, 0 to 11130
Data columns (total 12 columns):
 #   Column              Non-Null Count  Dtype 
---  ------              --------------  ----- 
 0   bookID              11131 non-null  object
 1   title               11131 non-null  str   
 2   authors             11131 non-null  str   
 3   average_rating      11131 non-null  object
 4   isbn                11131 non-null  str   
 5   isbn13              11131 non-null  object
 6   language_code       11131 non-null  str   
 7   num_pages           11131 non-null  object
 8   ratings_count       11131 non-null  object
 9   text_reviews_count  11131 non-null  object
 10  publication_date    11131 non-null  str   
 11  publisher           11131 non-null  str   
dtypes: object(6), str(6)
memory usage: 1.0+ MB


,bookID,title,authors,average_rating,isbn,isbn13,language_code,num_pages,ratings_count,text_reviews_count,publication_date,publisher
count,11131,11131,11131,11131.0,11131,11131,11131,11131,11131,11131,11131,11131
unique,11127,10352,6643,212.0,11127,11127,27,1001,5298,1825,3679,2292
top,12224,The Iliad,Stephen King,4.0,0674842111,9780674842113,eng,288,3,0,10/1/2005,Vintage
freq,2,9,40,219.0,2,2,8914,230,82,624,56,318


In [17]:
books.info()
books.describe()

<class 'pandas.DataFrame'>
RangeIndex: 11131 entries, 0 to 11130
Data columns (total 12 columns):
 #   Column              Non-Null Count  Dtype 
---  ------              --------------  ----- 
 0   bookID              11131 non-null  object
 1   title               11131 non-null  str   
 2   authors             11131 non-null  str   
 3   average_rating      11131 non-null  object
 4   isbn                11131 non-null  str   
 5   isbn13              11131 non-null  object
 6   language_code       11131 non-null  str   
 7   num_pages           11131 non-null  object
 8   ratings_count       11131 non-null  object
 9   text_reviews_count  11131 non-null  object
 10  publication_date    11131 non-null  str   
 11  publisher           11131 non-null  str   
dtypes: object(6), str(6)
memory usage: 1.0+ MB


,bookID,title,authors,average_rating,isbn,isbn13,language_code,num_pages,ratings_count,text_reviews_count,publication_date,publisher
count,11131,11131,11131,11131.0,11131,11131,11131,11131,11131,11131,11131,11131
unique,11127,10352,6643,212.0,11127,11127,27,1001,5298,1825,3679,2292
top,12224,The Iliad,Stephen King,4.0,0674842111,9780674842113,eng,288,3,0,10/1/2005,Vintage
freq,2,9,40,219.0,2,2,8914,230,82,624,56,318
